In [46]:
import importlib
import sys

if 'src.config' in sys.modules:
    del sys.modules['src.config']

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from category_encoders import TargetEncoder

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

sys.path.append('../')
from src.config import *

print(f"TRAIN_FILE = {TRAIN_FILE}")  

TRAIN_FILE = c:\Users\hp\Downloads\Explainable Credit Risk Scoring System\data\application_train\application_train.csv


IMPORTS

In [47]:
data=pd.read_csv(TRAIN_FILE)
print(f"Data shape: {data.shape}")

Data shape: (307511, 122)


Drop usless cols

In [48]:
msn_pct=data.isnull().mean()*100
msn_col_to_drop=msn_pct[msn_pct>60].index.tolist()
msn_col_to_drop.append(ID_COL)

data.drop(columns=msn_col_to_drop, inplace=True)
print(f"Data shape after dropping columns with >60% missing values: {data.shape}")

Data shape after dropping columns with >60% missing values: (307511, 104)


Dealin with outliers in DAYS_EMPLOYED

In [49]:
data['IS_UNEMPLOYED'] = (data['DAYS_EMPLOYED'] == 365243).astype(int)
data['DAYS_EMPLOYED'].replace(365243, np.nan, inplace=True)
print(f"Unemployed people flagged: {data['IS_UNEMPLOYED'].sum():,}")

Unemployed people flagged: 55,374


Features engineering

In [50]:
data['AGE_YEARS']=data['DAYS_BIRTH'] / -365
data['EMPLOYMENT_YEARS']=data['DAYS_EMPLOYED'] / -365
data['REGISTRATION_YEARS']=data['DAYS_REGISTRATION'] / -365
data['ID_PUBLISHER_YEARS']=data['DAYS_ID_PUBLISH'] / -365

data.drop(columns=['DAYS_BIRTH', 'DAYS_EMPLOYED','DAYS_REGISTRATION', 'DAYS_ID_PUBLISH'], inplace=True)

data['CREDIT_INCOME_RATIO']=data['AMT_CREDIT'] / data['AMT_INCOME_TOTAL']
data['ANNUITY_INCOME_RATIO']=data['AMT_ANNUITY'] / data['AMT_INCOME_TOTAL']
data['CREDIT_TERM']=data['AMT_ANNUITY'] / data['AMT_CREDIT']
data['INCOME_PER_PERSON']=data['AMT_INCOME_TOTAL'] / data['CNT_FAM_MEMBERS']
data['CHILDREN_RATIO']=data['CNT_CHILDREN'] / data['CNT_FAM_MEMBERS']

data['EXT_SOURCE_MEAN']=data[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].mean(axis=1)
data['EXT_SOURCE_MIN']  = data[['EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3']].min(axis=1)
data['EXT_SOURCE_STD']  = data[['EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3']].std(axis=1)


data['EMPLOYMENT_AGE_RATIO'] = data['EMPLOYMENT_YEARS'] / data['AGE_YEARS']

print(f"New shape after feature engineering: {data.shape}")


New shape after feature engineering: (307511, 114)


Vertical Split

In [51]:
X=data.drop(columns=['TARGET'])
y=data['TARGET']
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Default rate: {y.mean()*100:.2f}%")


X shape: (307511, 113)
y shape: (307511,)
Default rate: 8.07%


In [52]:
categorical_cols = X.select_dtypes(include='object').columns.tolist()
numeric_cols = X.select_dtypes(include=np.number).columns.tolist()

print(f"Categorical columns ({len(categorical_cols)}): {categorical_cols}")
print(f"Numeric columns: {len(numeric_cols)}")

Categorical columns (15): ['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE', 'WEEKDAY_APPR_PROCESS_START', 'ORGANIZATION_TYPE', 'HOUSETYPE_MODE', 'WALLSMATERIAL_MODE', 'EMERGENCYSTATE_MODE']
Numeric columns: 98


Horizental Split

In [53]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)
print(f"Training set shape: {X_train.shape}, {y_train.shape}")
print(f"Test set shape: {X_test.shape}, {y_test.shape}")

Training set shape: (246008, 113), (246008,)
Test set shape: (61503, 113), (61503,)


Imputation

In [54]:
num_imputer = SimpleImputer(strategy='median')
X_train[numeric_cols] = num_imputer.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = num_imputer.transform(X_test[numeric_cols])

cat_imputer = SimpleImputer(strategy='most_frequent')
X_train[categorical_cols] = cat_imputer.fit_transform(X_train[categorical_cols])
X_test[categorical_cols] = cat_imputer.transform(X_test[categorical_cols])


Encoding

In [55]:
encoder=TargetEncoder(cols=categorical_cols, smoothing=0.3)
X_train=encoder.fit_transform(X_train, y_train)
X_test=encoder.transform(X_test)
print(X_train.shape, X_test.shape)

(246008, 113) (61503, 113)


Scaling

In [56]:
scaler=StandardScaler()
X_train[numeric_cols]=scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols]=scaler.transform(X_test[numeric_cols])


In [57]:
X_train.to_csv('../data/X_train.csv', index=False)
X_test.to_csv('../data/X_test.csv', index=False)
y_train.to_csv('../data/y_train.csv', index=False)
y_test.to_csv('../data/y_test.csv', index=False)

# Save column lists for pipeline
joblib.dump(categorical_cols, '../models/categorical_cols.joblib')
joblib.dump(numeric_cols, '../models/numeric_cols.joblib')

print("All splits saved to data/")
print("Column lists saved to models/")

All splits saved to data/
Column lists saved to models/
